## LDA Topic Modeling

This notebook keeps the durable analysis path:

1. Prepare and normalize the Greek corpus.
2. Evaluate candidate topic counts and choose the final model.
3. Split broad topics into subtopics.
4. Visualize the final topic structure in 3D.
5. Attach English glosses for interpretable topic labels.

In [19]:
import pandas as pd

df = pd.read_csv("data/nt_split_column_df.csv")
df.head()

,verse_ref,text,book,chapter,verse
0,1 corinthians 1:1,Παῦλος κλητὸς ἀπόστολος Χριστοῦ Ἰησοῦ διὰ θελή...,1 corinthians,1,1
1,1 corinthians 1:2,"τῇ ἐκκλησίᾳ τοῦ θεοῦ τῇ οὔσῃ ἐν Κορίνθῳ, ἡγι...",1 corinthians,1,2
2,1 corinthians 1:3,χάρις ὑμῖν καὶ εἰρήνη ἀπὸ θεοῦ πατρὸς ἡμῶν καὶ...,1 corinthians,1,3
3,1 corinthians 1:4,Εὐχαριστῶ τῷ θεῷ μου πάντοτε περὶ ὑμῶν ἐπὶ τῇ...,1 corinthians,1,4
4,1 corinthians 1:5,ὅτι ἐν παντὶ ἐπλουτίσθητε ἐν αὐτῷ ἐν παντὶ λόγ...,1 corinthians,1,5


In [20]:
import re
import unicodedata
from collections import Counter


def strip_diacritics(text):
    decomposed = unicodedata.normalize("NFD", text)
    return "".join(char for char in decomposed if unicodedata.category(char) != "Mn")


RAW_GREEK_STOPWORDS = {
    "και", "δε", "γαρ", "ουν", "ως", "μη", "ου", "ουκ", "ουχ",
    "ο", "η", "το", "οι", "αι", "τα",
    "του", "της", "των", "τω", "τη", "τον", "την", "τους", "τας",
    "τοις", "ταις",
    "εν", "εις", "εκ", "εξ", "επι", "δια", "κατα", "προς", "υπο",
    "υπερ", "μετα", "παρα", "απο", "περι",
    "οτι", "ινα", "εαν", "αν",
    "τις", "τι", "τινα", "τινες",
    "αυτου", "αυτων", "αυτον", "αυτη", "αυτης", "αυται", "αυτοι", "αυτω",
    "ημιν", "ημων", "ημας", "υμιν", "υμων", "υμας",
    "μου", "σου", "με", "σε"
}
GREEK_STOPWORDS = {
    strip_diacritics(word.lower()).replace("ς", "σ") for word in RAW_GREEK_STOPWORDS
}

RAW_STEM_SUFFIXES = {
    "ομεθα", "ουμεθα", "εσθαι", "ησονται", "ονται", "ουσιν", "ουσι",
    "ομεν", "ουμεν", "ειτε", "ετε", "εται", "ειται", "ειν", "εις",
    "οντες", "οντας", "μενος", "μενη", "μενον", "ματος", "ματων",
    "μασι", "σεως", "σεων", "σεσι", "τητες", "τητων", "τητα",
    "ικος", "ικου", "ικον", "ικοι", "ικων",
    "οις", "αις", "ους", "ων", "ου", "ος", "ον", "οι", "αι",
    "ας", "ες", "ης", "ην", "ει"
}
STEM_SUFFIXES = sorted(
    {suffix.replace("ς", "σ") for suffix in RAW_STEM_SUFFIXES},
    key=len,
    reverse=True,
)


def normalize_greek(text):
    text = strip_diacritics(str(text).lower()).replace("ς", "σ")
    text = re.sub(r"[^α-ω\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def light_stem_greek_token(token):
    if len(token) <= 4:
        return token

    for suffix in STEM_SUFFIXES:
        if token.endswith(suffix) and len(token) - len(suffix) >= 3:
            return token[:-len(suffix)]

    return token


def preprocess_document(text):
    normalized = normalize_greek(text)
    tokens = [
        token
        for token in normalized.split()
        if len(token) > 2 and token not in GREEK_STOPWORDS
    ]
    stems = [light_stem_greek_token(token) for token in tokens]
    stems = [stem for stem in stems if len(stem) > 1]
    return normalized, tokens, stems


df[["normalized_text", "tokens", "stemmed_tokens"]] = df["text"].apply(
    lambda text: pd.Series(preprocess_document(text))
)
corpus = df["stemmed_tokens"].tolist()
df["corpus_text"] = df["stemmed_tokens"].str.join(" ")

top_stems = Counter(stem for doc in corpus for stem in doc).most_common(20)
pd.DataFrame(top_stems, columns=["stem", "count"])

,stem,count
0,αυτ,1304
1,εστιν,934
2,λεγ,867
3,ιησ,859
4,θεου,704
5,ειπεν,626
6,κυρι,527
7,ανθρωπ,525
8,αλλ,470
9,χριστ,469


## Model Selection

The next cells build the document-term matrix, compare several topic counts, and keep the best-performing LDA model for the rest of the notebook.

In [21]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=5,
    max_df=0.45,
    max_features=3000,
)

doc_term_matrix = vectorizer.fit_transform(df["corpus_text"])
feature_names = vectorizer.get_feature_names_out()

print(doc_term_matrix.shape)
df[["verse_ref", "stemmed_tokens", "corpus_text"]].head()

(7958, 2422)


,verse_ref,stemmed_tokens,corpus_text
0,1 corinthians 1:1,"[παυλ, κλητ, αποστολ, χριστ, ιησ, θελη, θεου, ...",παυλ κλητ αποστολ χριστ ιησ θελη θεου σωσθεν α...
1,1 corinthians 1:2,"[εκκλησια, θεου, ουση, κορινθω, ηγιασμεν, χρισ...",εκκλησια θεου ουση κορινθω ηγιασμεν χριστω ιησ...
2,1 corinthians 1:3,"[χαρισ, ειρηνη, θεου, πατρ, κυρι, ιησ, χριστ]",χαρισ ειρηνη θεου πατρ κυρι ιησ χριστ
3,1 corinthians 1:4,"[ευχαριστω, θεω, παντοτε, χαριτι, θεου, δοθεισ...",ευχαριστω θεω παντοτε χαριτι θεου δοθειση χρισ...
4,1 corinthians 1:5,"[παντι, επλουτισθητε, παντι, λογω, παση, γνωσ]",παντι επλουτισθητε παντι λογω παση γνωσ


In [22]:
candidate_topics = [5, 8, 10, 12, 15, 20, 25, 30, 40, 50]
lda_results = []
lda_models = {}

for k in candidate_topics:
    model = LatentDirichletAllocation(
        n_components=k,
        learning_method="batch",
        max_iter=8,
        random_state=42,
    )
    doc_topic = model.fit_transform(doc_term_matrix)
    lda_models[k] = (model, doc_topic)
    lda_results.append(
        {
            "n_topics": k,
            "perplexity": model.perplexity(doc_term_matrix),
            "log_likelihood": model.score(doc_term_matrix),
            "avg_topic_confidence": doc_topic.max(axis=1).mean(),
        }
    )

lda_eval_df = (
    pd.DataFrame(lda_results)
    .sort_values("perplexity")
    .reset_index(drop=True)
)
lda_eval_df

,n_topics,perplexity,log_likelihood,avg_topic_confidence
0,5,1306.368387,-442934.666583,0.711946
1,8,1380.309203,-446333.467656,0.682188
2,10,1443.359293,-449090.815611,0.670214
3,12,1479.932533,-450635.576644,0.649557
4,15,1513.637835,-452025.767656,0.622677
5,20,1626.679969,-456472.098916,0.605668
6,25,1727.750713,-460193.320177,0.590280
7,30,1777.040482,-461929.803626,0.572857
8,40,1990.487560,-468932.195336,0.552086
9,50,2135.232858,-473265.610812,0.532171


In [23]:
best_k = int(lda_eval_df.loc[0, "n_topics"])
best_lda, best_topic_matrix = lda_models[best_k]

best_topics_df = describe_topics(best_lda, feature_names, n_words=12)
df["best_dominant_topic"] = best_topic_matrix.argmax(axis=1)
df["best_topic_confidence"] = best_topic_matrix.max(axis=1)

print(f"Selected topic count: {best_k}")
best_topics_df

Selected topic count: 5


,topic,top_terms
0,0,"εαυτ, αλλα, παντ, μεν, αυτ, πασ, υμεισ, ουτωσ,..."
1,1,"αυτ, εστιν, ανθρωπ, κυρι, ουτ, λεγ, παντ, ειπε..."
2,2,"εστιν, θεου, χριστ, αλλ, ιησ, κυρι, παντα, ανθ..."
3,3,"αυτ, λεγ, ιδου, πολλ, αγγελ, ανθρωπ, ημερ, παν..."
4,4,"ιησ, ειπεν, αυτ, λεγ, εγω, μαθητ, πετρ, ειμι, ..."


## Hierarchical Subtopics

After selecting the best topic count, this section splits each parent topic into smaller subtopics and summarizes the resulting topic paths.

In [24]:
parent_topic_col = "best_dominant_topic"
parent_conf_col = "best_topic_confidence"

target_subtopics_per_parent = 4
min_docs_to_split = 120
min_docs_per_subtopic = 60
max_subtopics_per_parent = 6
subtopic_random_state = 42

parent_topic_terms = best_topics_df.set_index("topic")["top_terms"].to_dict()

df["subtopic"] = 0
df["subtopic_confidence"] = df[parent_conf_col]
df["topic_path"] = df[parent_topic_col].astype(int).astype(str) + ".0"

subtopic_models = {}
subtopic_rows = []

for parent_topic in sorted(df[parent_topic_col].unique()):
    mask = df[parent_topic_col] == parent_topic
    parent_idx = df.index[mask].to_numpy()
    n_docs = parent_idx.size

    if n_docs < min_docs_to_split:
        subtopic_rows.append(
            {
                "parent_topic": int(parent_topic),
                "subtopic": 0,
                "topic_path": f"{int(parent_topic)}.0",
                "doc_count": int(n_docs),
                "avg_confidence": float(df.loc[mask, parent_conf_col].mean()),
                "top_terms": parent_topic_terms.get(int(parent_topic), ""),
                "split_applied": False,
            }
        )
        continue

    k_by_size = max(2, n_docs // min_docs_per_subtopic)
    n_subtopics = min(max_subtopics_per_parent, target_subtopics_per_parent, k_by_size)
    n_subtopics = min(n_subtopics, max(2, n_docs - 1))

    parent_doc_term_matrix = doc_term_matrix[parent_idx]
    sub_lda = LatentDirichletAllocation(
        n_components=n_subtopics,
        learning_method="batch",
        max_iter=15,
        random_state=subtopic_random_state,
    )
    sub_doc_topic = sub_lda.fit_transform(parent_doc_term_matrix)

    sub_ids = sub_doc_topic.argmax(axis=1)
    sub_conf = sub_doc_topic.max(axis=1)

    df.loc[parent_idx, "subtopic"] = sub_ids
    df.loc[parent_idx, "subtopic_confidence"] = sub_conf
    df.loc[parent_idx, "topic_path"] = [f"{int(parent_topic)}.{int(s)}" for s in sub_ids]

    subtopic_models[int(parent_topic)] = {
        "model": sub_lda,
        "doc_indices": parent_idx,
        "doc_topic": sub_doc_topic,
    }

    subtopic_terms_df = describe_topics(sub_lda, feature_names, n_words=10)
    for row in subtopic_terms_df.itertuples(index=False):
        sub_mask = sub_ids == row.topic
        subtopic_rows.append(
            {
                "parent_topic": int(parent_topic),
                "subtopic": int(row.topic),
                "topic_path": f"{int(parent_topic)}.{int(row.topic)}",
                "doc_count": int(sub_mask.sum()),
                "avg_confidence": float(sub_conf[sub_mask].mean()),
                "top_terms": row.top_terms,
                "split_applied": True,
            }
        )

subtopic_summary_df = (
    pd.DataFrame(subtopic_rows)
    .sort_values(["parent_topic", "subtopic"])
    .reset_index(drop=True)
)

print(f"Selected {best_k} parent topics")
print(f"Discovered {df['topic_path'].nunique()} topic paths")
subtopic_summary_df

Selected 5 parent topics
Discovered 20 topic paths


,parent_topic,subtopic,topic_path,doc_count,avg_confidence,top_terms,split_applied
0,0,0,0.0,648,0.799467,"εαυτ, μεν, αλλα, αυτ, πασ, υμεισ, θεοσ, ποι, α...",True
1,0,1,0.1,247,0.800807,"παντ, εαυτ, λαλ, καθωσ, θεου, αυτ, ανθρωπ, δικ...",True
2,0,2,0.2,136,0.817519,"αυτ, ονοματι, εστιν, θεου, λεγ, δενδρ, καρπ, κ...",True
3,0,3,0.3,271,0.795842,"ειτε, αιων, θεω, αμην, παντ, δοξα, θεοσ, μαλλ,...",True
4,1,0,1.0,172,0.762997,"φαρισαι, αυτ, ταυτα, εγενετο, ειπαν, ποι, σαββ...",True
5,1,1,1.1,952,0.811713,"αυτ, εστιν, ιησ, ανθρωπ, κυρι, λεγ, παντ, ειπε...",True
6,1,2,1.2,165,0.753465,"νομ, εστιν, αμαρτι, ιουδαι, αβρααμ, αιματ, ιμα...",True
7,1,3,1.3,262,0.790177,"προφητ, αυτ, νομ, ειπεν, λεγ, κριν, παντ, κυρι...",True
8,2,0,2.0,181,0.760244,"εστιν, αληθει, θεου, θεοσ, αυτ, μεν, αλλ, κοσμ...",True
9,2,1,2.1,436,0.746932,"χριστ, θεου, ιησ, κυρι, εστιν, αλλ, ανθρωπ, χα...",True


## Final Visualization

Keep one chart that shows the final topic geometry after model selection and subtopic assignment.

In [25]:
import plotly.graph_objects as go
import plotly.express as px
from sklearn.decomposition import PCA

X = best_topic_matrix
coords3d = PCA(n_components=3, random_state=42).fit_transform(X)
df["cluster_x3"] = coords3d[:, 0]
df["cluster_y3"] = coords3d[:, 1]
df["cluster_z3"] = coords3d[:, 2]

if "best_dominant_topic" not in df.columns:
    df["best_dominant_topic"] = best_topic_matrix.argmax(axis=1)
if "best_topic_confidence" not in df.columns:
    df["best_topic_confidence"] = best_topic_matrix.max(axis=1)

topic_path_col = "topic_path" if "topic_path" in df.columns else "best_dominant_topic"
conf_col = "subtopic_confidence" if "subtopic_confidence" in df.columns else "best_topic_confidence"

df["hover_text_3d"] = (
    "<b>" + df["verse_ref"].astype(str) + "</b><br>"
    + df["text"].str[:120] + "...<br>"
    + "Topic: " + df["best_dominant_topic"].astype(str) + "<br>"
    + "Path: " + df[topic_path_col].astype(str) + "<br>"
    + "Confidence: " + df[conf_col].round(3).astype(str)
)

palette = px.colors.qualitative.Set2
fig = go.Figure()

for i, topic_id in enumerate(sorted(df["best_dominant_topic"].unique())):
    topic_docs = df[df["best_dominant_topic"] == topic_id]
    color = palette[i % len(palette)]
    lg = f"topic_{topic_id}"

    fig.add_trace(
        go.Scatter3d(
            x=topic_docs["cluster_x3"],
            y=topic_docs["cluster_y3"],
            z=topic_docs["cluster_z3"],
            mode="markers",
            name=f"T{topic_id}",
            legendgroup=lg,
            marker=dict(size=3.5, color=color, opacity=0.55),
            customdata=topic_docs[["hover_text_3d"]].values,
            hovertemplate="%{customdata[0]}<extra></extra>",
        )
    )

    cx = float(topic_docs["cluster_x3"].mean())
    cy = float(topic_docs["cluster_y3"].mean())
    cz = float(topic_docs["cluster_z3"].mean())

    top_docs = topic_docs.nlargest(8, "best_topic_confidence")
    for _, doc_row in top_docs.iterrows():
        fig.add_trace(
            go.Scatter3d(
                x=[cx, float(doc_row["cluster_x3"])],
                y=[cy, float(doc_row["cluster_y3"])],
                z=[cz, float(doc_row["cluster_z3"])],
                mode="lines",
                legendgroup=lg,
                showlegend=False,
                line=dict(color=color, width=2),
                hoverinfo="skip",
            )
        )

    fig.add_trace(
        go.Scatter3d(
            x=[cx], y=[cy], z=[cz],
            mode="markers+text",
            legendgroup=lg,
            showlegend=False,
            marker=dict(size=10, color=color, line=dict(color="black", width=1)),
            text=[f"T{topic_id}"],
            textposition="top center",
            hovertemplate=f"Topic {topic_id} centroid<extra></extra>",
        )
    )

fig.update_layout(
    title="LDA Topics in 3D",
    height=780,
    legend_title_text="Topic",
    scene=dict(
        xaxis_title="PC1",
        yaxis_title="PC2",
        zaxis_title="PC3",
    ),
)

fig.show()

## English Gloss Interpretation

The last section translates topic structure into English-facing labels, example verses, and lookup maps that can be reused in the UI.

In [30]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

# --- Deterministic English gloss support from og_lang_transformed.csv ---
translation_df = pd.read_csv("data/og_lang_transformed.csv")
translation_df = translation_df.dropna(subset=["book", "chapter", "verse", "translation"]).copy()


def _normalize_book_name(book_name):
    return re.sub(r"\s+", " ", str(book_name).strip().lower())


def _verse_ref_key(book_name, chapter, verse):
    return f"{_normalize_book_name(book_name)} {int(chapter)}:{int(verse)}"


translation_df["verse_ref_key"] = translation_df.apply(
    lambda row: _verse_ref_key(row["book"], row["chapter"], row["verse"]),
    axis=1,
)

# KJV verse text map for English snippets aligned to verse_ref
kjv_df = pd.read_csv("data/kjv.csv")
kjv_df = kjv_df.dropna(subset=["book_name", "chapter_number", "verse_number", "verse_text"]).copy()
kjv_df["verse_ref_key"] = kjv_df.apply(
    lambda row: _verse_ref_key(row["book_name"], row["chapter_number"], row["verse_number"]),
    axis=1,
)
kjv_verse_map = (
    kjv_df[["verse_ref_key", "verse_text"]]
    .drop_duplicates(subset=["verse_ref_key"], keep="first")
    .set_index("verse_ref_key")["verse_text"]
    .to_dict()
)

english_stopwords = {
    "the", "and", "of", "to", "in", "a", "an", "for", "on", "with", "from",
    "by", "at", "into", "through", "that", "this", "these", "those", "then",
    "him", "her", "them", "his", "their", "its", "you", "your", "our", "us",
    "he", "she", "they", "it", "was", "is", "are", "be", "being", "been",
}


def _clean_gloss(gloss):
    gloss = str(gloss).lower().strip()
    gloss = re.sub(r"[\[\]<>]", "", gloss)
    gloss = re.sub(r"[^a-z\s'-]", " ", gloss)
    gloss = re.sub(r"\s+", " ", gloss).strip()
    if not gloss or gloss in english_stopwords:
        return ""
    return gloss


translation_df["translation_clean"] = translation_df["translation"].map(_clean_gloss)
translation_df = translation_df[translation_df["translation_clean"] != ""].copy()

# Aggregate deterministic English glosses per verse
verse_gloss_map = (
    translation_df.groupby("verse_ref_key")["translation_clean"]
    .apply(list)
    .to_dict()
)

# Add a normalized key to the LDA dataframe
df["verse_ref_key"] = df["verse_ref"].astype(str).str.strip().str.lower()
df["english_glosses"] = df["verse_ref_key"].map(lambda key: verse_gloss_map.get(key, []))


# --- Helpers for readable topic descriptions ---
def _short_label(terms_str, k=3):
    terms = [t.strip() for t in str(terms_str).split(",") if t.strip()]
    return " / ".join(terms[:k])


def _top_refs_for_path(topic_path, n=2):
    rows = (
        df[df["topic_path"] == topic_path]
        .sort_values("subtopic_confidence", ascending=False)
        [["verse_ref", "text", "subtopic_confidence"]]
        .head(n)
        .copy()
    )
    return rows


def _truncate(text, n=130):
    text = str(text)
    return text if len(text) <= n else text[:n] + "..."


def _top_english_glosses(frame, top_n=6):
    gloss_counter = Counter()
    for gloss_list in frame.get("english_glosses", []):
        gloss_counter.update(gloss_list)

    ranked = []
    seen = set()
    for gloss, _count in gloss_counter.most_common():
        base = gloss.strip()
        if not base or base in seen:
            continue
        seen.add(base)
        ranked.append(base)
        if len(ranked) >= top_n:
            break

    return ", ".join(ranked)


def _greek_snippets(refs, greek_texts, greek_n=110):
    rows = []
    for ref, greek in zip(refs, greek_texts):
        rows.append(f"{ref} | {_truncate(greek, greek_n)}")
    return " || ".join(rows)


def _english_snippets(refs, eng_n=110):
    rows = []
    for ref in refs:
        key = str(ref).strip().lower()
        english = kjv_verse_map.get(key, "")
        english_snippet = _truncate(english, eng_n) if english else "[No KJV verse found]"
        rows.append(f"{ref} | {english_snippet}")
    return " || ".join(rows)


if "best_lda" not in globals() or "best_topics_df" not in globals():
    raise RuntimeError("Run the LDA model-selection cells first so best_lda and best_topics_df are available.")

# --- Topic-level explainability (global topics) ---
topic_sizes = df["best_dominant_topic"].value_counts().sort_index()
topic_avg_conf = df.groupby("best_dominant_topic")["best_topic_confidence"].mean().sort_index()

# Distinctive Greek terms via simple lift: p(term|topic) / p(term|all topics)
topic_term = best_lda.components_ / best_lda.components_.sum(axis=1, keepdims=True)
global_term = topic_term.mean(axis=0, keepdims=True)
lift = np.log((topic_term + 1e-12) / (global_term + 1e-12))

topic_cards = []
for t in sorted(best_topics_df["topic"].tolist()):
    topic_frame = df[df["best_dominant_topic"] == t]
    top_lift_idx = lift[t].argsort()[-6:][::-1]
    distinctive_terms = ", ".join(feature_names[i] for i in top_lift_idx)
    top_terms = best_topics_df.loc[best_topics_df["topic"] == t, "top_terms"].iloc[0]
    english_equivalents = _top_english_glosses(topic_frame, top_n=6)

    reps = (
        topic_frame
        .sort_values("best_topic_confidence", ascending=False)
        [["verse_ref", "text", "best_topic_confidence"]]
        .head(2)
    )
    refs = reps["verse_ref"].astype(str).tolist()
    refs_text = " | ".join(refs)
    greek_snippets = _greek_snippets(refs, reps["text"].tolist(), greek_n=110)
    english_snippets = _english_snippets(refs, eng_n=110)

    topic_cards.append(
        {
            "topic": int(t),
            "label": _short_label(top_terms, k=3),
            "english_equivalents": english_equivalents,
            "doc_count": int(topic_sizes.get(t, 0)),
            "avg_conf": round(float(topic_avg_conf.get(t, 0.0)), 3),
            "top_terms": top_terms,
            "distinctive_terms": distinctive_terms,
            "example_refs": refs_text,
            "example_snippets": greek_snippets,
            "english_example_snippets": english_snippets,
        }
    )

topic_interpretation_df = pd.DataFrame(topic_cards).sort_values("topic").reset_index(drop=True)
print(f"Topic interpretation cards: {len(topic_interpretation_df)}")
display(topic_interpretation_df)

# --- Subtopic-level explainability (hierarchical topic paths) ---
if "topic_path" not in df.columns or "subtopic_summary_df" not in globals():
    print("Subtopic data not found. Run the hierarchical subtopic cell first.")
else:
    subtopic_cards = []
    sub_df = subtopic_summary_df.copy().sort_values(["parent_topic", "subtopic"]).reset_index(drop=True)

    for row in sub_df.itertuples(index=False):
        path = str(row.topic_path)
        path_frame = df[df["topic_path"] == path]
        reps = _top_refs_for_path(path, n=2)
        refs = reps["verse_ref"].astype(str).tolist() if len(reps) else []
        refs_text = " | ".join(refs) if refs else ""
        greek_snippets = _greek_snippets(refs, reps["text"].tolist(), greek_n=100) if len(reps) else ""
        english_snippets = _english_snippets(refs, eng_n=100) if len(reps) else ""
        english_equivalents = _top_english_glosses(path_frame, top_n=6)

        parent_terms = best_topics_df.loc[best_topics_df["topic"] == int(row.parent_topic), "top_terms"].iloc[0]
        parent_label = _short_label(parent_terms, k=2)
        sub_label = _short_label(row.top_terms, k=2)
        combined_label = f"{parent_label} -> {sub_label}"

        subtopic_cards.append(
            {
                "topic_path": path,
                "parent_topic": int(row.parent_topic),
                "subtopic": int(row.subtopic),
                "label": combined_label,
                "english_equivalents": english_equivalents,
                "doc_count": int(row.doc_count),
                "avg_conf": round(float(row.avg_confidence), 3),
                "top_terms": row.top_terms,
                "example_refs": refs_text,
                "example_snippets": greek_snippets,
                "english_example_snippets": english_snippets,
            }
        )

    subtopic_interpretation_df = pd.DataFrame(subtopic_cards)
    print(f"Subtopic interpretation cards: {len(subtopic_interpretation_df)}")
    display(subtopic_interpretation_df.head(30))

# Optional maps to plug into a UI tooltip layer
topic_label_map = dict(zip(topic_interpretation_df["topic"], topic_interpretation_df["label"]))
if "topic_path" in df.columns and "subtopic_interpretation_df" in globals():
    subtopic_label_map = dict(zip(subtopic_interpretation_df["topic_path"], subtopic_interpretation_df["label"]))

Topic interpretation cards: 5


,topic,label,english_equivalents,doc_count,avg_conf,top_terms,distinctive_terms,example_refs,example_snippets,english_example_snippets
0,0,εαυτ / αλλα / παντ,"not, of the, however, now, but, also",1302,0.695,"εαυτ, αλλα, παντ, μεν, αυτ, πασ, υμεισ, ουτωσ,...","ειτε, δοξα, αιωνα, ιδιαν, γινεσθε, δενδρ",matthew 5:22 | revelation 22:2,matthew 5:22 | ἐγὼ δὲ λέγω ὑμῖν ὅτι πᾶς ὁ ὀργι...,"matthew 5:22 | But I say unto you, That whosoe..."
1,1,αυτ / εστιν / ανθρωπ,"not, of the, of him, now, all, jesus",1551,0.725,"αυτ, εστιν, ανθρωπ, κυρι, ουτ, λεγ, παντ, ειπε...","σαββατ, ερημω, μαριαμ, γενε, απολεσ, γεωργ",acts 21:28 | acts 9:21,"acts 21:28 | κράζοντες· ἄνδρες Ἰσραηλῖται, βοη...","acts 21:28 | Crying out, Men of Israel, help: ..."
2,2,εστιν / θεου / χριστ,"not, of the, now, god, but, christ",1731,0.712,"εστιν, θεου, χριστ, αλλ, ιησ, κυρι, παντα, ανθ...","ουτε, αληθει, σοφια, αληθ, σωτηρ, γνω",romans 11:6 | matthew 5:19,"romans 11:6 | εἰ δὲ χάριτι οὐκέτι ἐξ ἔργων, ἐπ...","romans 11:6 | And if by grace, then is it no m..."
3,3,αυτ / λεγ / ιδου,"of the, not, of him, now, who, of you",1591,0.714,"αυτ, λεγ, ιδου, πολλ, αγγελ, ανθρωπ, ημερ, παν...","εθνη, εγεννησεν, ηκουσα, μητε, καθη, στομα",revelation 17:1 | revelation 7:2,revelation 17:1 | Καὶ ἦλθεν εἷς ἐκ τῶν ἑπτὰ ἀγ...,revelation 17:1 | And there came one of the se...
4,4,ιησ / ειπεν / αυτ,"jesus, not, now, of the, of him, to him",1783,0.711,"ιησ, ειπεν, αυτ, λεγ, εγω, μαθητ, πετρ, ειμι, ...","ειμι, πετρ, σιμ, εγειρ, τιμοθε, περαν",luke 9:33 | luke 7:22,luke 9:33 | Καὶ ἐγένετο ἐν τῷ διαχωρίζεσθαι αὐ...,"luke 9:33 | And it came to pass, as they depar..."


Subtopic interpretation cards: 20


,topic_path,parent_topic,subtopic,label,english_equivalents,doc_count,avg_conf,top_terms,example_refs,example_snippets,english_example_snippets
0,0.0,0,0,εαυτ / αλλα -> εαυτ / μεν,"not, of the, but, however, also, of him",648,0.799,"εαυτ, μεν, αλλα, αυτ, πασ, υμεισ, θεοσ, ποι, α...",matthew 12:45 | acts 22:3,matthew 12:45 | τότε πορεύεται καὶ παραλαμβάνε...,"matthew 12:45 | Then goeth he, and taketh with..."
1,0.1,0,1,εαυτ / αλλα -> παντ / εαυτ,"not, of the, now, all, if, however",247,0.801,"παντ, εαυτ, λαλ, καθωσ, θεου, αυτ, ανθρωπ, δικ...",mark 6:11 | luke 6:45,mark 6:11 | καὶ ὃς ἂν τόπος μὴ δέξηται ὑμᾶς μη...,mark 6:11 | And whosoever shall not receive yo...
2,0.2,0,2,εαυτ / αλλα -> αυτ / ονοματι,"not, of the, now, of you, which, who",136,0.818,"αυτ, ονοματι, εστιν, θεου, λεγ, δενδρ, καρπ, κ...",luke 9:48 | acts 21:24,luke 9:48 | καὶ εἶπεν αὐτοῖς· ὃς ἐὰν δέξηται τ...,"luke 9:48 | And said unto them, Whosoever shal..."
3,0.3,0,3,εαυτ / αλλα -> ειτε / αιων,"not, of the, god, now, all, of him",271,0.796,"ειτε, αιων, θεω, αμην, παντ, δοξα, θεοσ, μαλλ,...",revelation 5:13 | revelation 2:13,revelation 5:13 | καὶ πᾶν κτίσμα ὃ ἐστιν ἐν τῷ...,revelation 5:13 | And every creature which is ...
4,1.0,1,0,αυτ / εστιν -> φαρισαι / αυτ,"of the, not, of him, these things, now, pharisees",172,0.763,"φαρισαι, αυτ, ταυτα, εγενετο, ειπαν, ποι, σαββ...",luke 18:11 | john 19:11,luke 18:11 | ὁ Φαρισαῖος σταθεὶς πρὸς ἑαυτὸν τ...,luke 18:11 | The Pharisee stood and prayed thu...
5,1.1,1,1,αυτ / εστιν -> αυτ / εστιν,"not, of the, jesus, all, of him, now",952,0.812,"αυτ, εστιν, ιησ, ανθρωπ, κυρι, λεγ, παντ, ειπε...",acts 14:15 | acts 4:10,"acts 14:15 | καὶ λέγοντες· ἄνδρες, τί ταῦτα πο...","acts 14:15 | And saying, Sirs, why do ye these..."
6,1.2,1,2,αυτ / εστιν -> νομ / εστιν,"not, of him, of the, law, however, of you",165,0.753,"νομ, εστιν, αμαρτι, ιουδαι, αβρααμ, αιματ, ιμα...",revelation 16:19 | matthew 23:35,revelation 16:19 | καὶ ἐγένετο ἡ πόλις ἡ μεγάλ...,revelation 16:19 | And the great city was divi...
7,1.3,1,3,αυτ / εστιν -> προφητ / αυτ,"not, of the, all, now, of you, however",262,0.790,"προφητ, αυτ, νομ, ειπεν, λεγ, κριν, παντ, κυρι...",1 corinthians 14:26 | mark 2:19,"1 corinthians 14:26 | Τί οὖν ἐστιν, ἀδελφοί; ὅ...","1 corinthians 14:26 | How is it then, brethren..."
8,2.0,2,0,εστιν / θεου -> εστιν / αληθει,"not, god, of the, but, now, however",181,0.760,"εστιν, αληθει, θεου, θεοσ, αυτ, μεν, αλλ, κοσμ...",john 5:30 | john 15:26,john 5:30 | οὐ δύναμαι ἐγὼ ποιεῖν ἀπ᾽ ἐμαυτοῦ ...,john 5:30 | I can of mine own self do nothing:...
9,2.1,2,1,εστιν / θεου -> χριστ / θεου,"not, of the, christ, god, of god, jesus",436,0.747,"χριστ, θεου, ιησ, κυρι, εστιν, αλλ, ανθρωπ, χα...",1 john 5:6 | 1 thessalonians 1:1,1 john 5:6 | οὗτός ἐστιν ὁ ἐλθὼν δι᾽ ὕδατος κα...,1 john 5:6 | This is he that came by water and...


In [31]:
subtopic_interpretation_df[["example_snippets", "english_example_snippets"]].iloc[0]

example_snippets            matthew 12:45 | τότε πορεύεται καὶ παραλαμβάνε...
english_example_snippets    matthew 12:45 | Then goeth he, and taketh with...
Name: 0, dtype: object